In [1]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
DATA_PATH = "../data/raw"

files = glob.glob(os.path.join(DATA_PATH, "*.csv"))

print(f"Found {len(files)} CSV files:\n")

for file in files:
    print(os.path.basename(file))


Found 8 CSV files:

Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv


In [3]:
monday_file = [
    f for f in files
    if "Monday" in os.path.basename(f)
][0]

print(monday_file)

../data/raw/Monday-WorkingHours.pcap_ISCX.csv


In [4]:
df = pd.read_csv(monday_file)

print("Original shape:", df.shape)

Original shape: (529918, 79)


In [5]:
df.columns = df.columns.str.strip()

In [6]:
df.columns.tolist()

['Destination Port',
 'Flow Duration',
 'Total Fwd Packets',
 'Total Backward Packets',
 'Total Length of Fwd Packets',
 'Total Length of Bwd Packets',
 'Fwd Packet Length Max',
 'Fwd Packet Length Min',
 'Fwd Packet Length Mean',
 'Fwd Packet Length Std',
 'Bwd Packet Length Max',
 'Bwd Packet Length Min',
 'Bwd Packet Length Mean',
 'Bwd Packet Length Std',
 'Flow Bytes/s',
 'Flow Packets/s',
 'Flow IAT Mean',
 'Flow IAT Std',
 'Flow IAT Max',
 'Flow IAT Min',
 'Fwd IAT Total',
 'Fwd IAT Mean',
 'Fwd IAT Std',
 'Fwd IAT Max',
 'Fwd IAT Min',
 'Bwd IAT Total',
 'Bwd IAT Mean',
 'Bwd IAT Std',
 'Bwd IAT Max',
 'Bwd IAT Min',
 'Fwd PSH Flags',
 'Bwd PSH Flags',
 'Fwd URG Flags',
 'Bwd URG Flags',
 'Fwd Header Length',
 'Bwd Header Length',
 'Fwd Packets/s',
 'Bwd Packets/s',
 'Min Packet Length',
 'Max Packet Length',
 'Packet Length Mean',
 'Packet Length Std',
 'Packet Length Variance',
 'FIN Flag Count',
 'SYN Flag Count',
 'RST Flag Count',
 'PSH Flag Count',
 'ACK Flag Count',
 'UR

In [7]:
df = df.replace([np.inf, -np.inf], np.nan)

In [8]:
print("Total missing values:", df.isnull().sum().sum())

Total missing values: 874


In [9]:
missing = df.isnull().sum()

missing[missing > 0].sort_values(ascending=False)

Flow Bytes/s      437
Flow Packets/s    437
dtype: int64

In [10]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Rows before: 529918
Rows after: 502983
Duplicates removed: 26935


In [11]:
X = df.drop(columns=["Label"])
y = df["Label"]

In [12]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (502983, 78)
y shape: (502983,)


In [13]:
y.value_counts()

Label
BENIGN    502983
Name: count, dtype: int64

In [14]:
def clean_dataframe(df):
    # 1. Clean column names
    df.columns = df.columns.str.strip()

    # 2. Convert infinite values to NaN
    df = df.replace([np.inf, -np.inf], np.nan)

    # 3. Remove exact duplicate rows
    df = df.drop_duplicates()

    return df

In [15]:
def map_attack_label(label):
    label = str(label).strip()

    if label == "BENIGN":
        return "BENIGN"

    if label.startswith("DoS "):
        return "DoS"

    if label == "DDoS":
        return "DDoS"

    if label == "PortScan":
        return "PortScan"

    if label in ["FTP-Patator", "SSH-Patator"]:
        return "Brute Force"

    if "Web Attack" in label and "Brute Force" in label:
        return "Brute Force"

    if label == "Bot":
        return "Bot"

    if "Web Attack" in label:
        return "Web Attack"

    return "EXCLUDE"

In [16]:
test_labels = [
    "BENIGN",
    "DoS Hulk",
    "DoS GoldenEye",
    "DoS slowloris",
    "DoS Slowhttptest",
    "DDoS",
    "PortScan",
    "FTP-Patator",
    "SSH-Patator",
    "Bot",
    "Web Attack � Brute Force",
    "Web Attack � XSS",
    "Web Attack � Sql Injection",
    "Infiltration",
    "Heartbleed"
]

for label in test_labels:
    print(f"{label:40} → {map_attack_label(label)}")

BENIGN                                   → BENIGN
DoS Hulk                                 → DoS
DoS GoldenEye                            → DoS
DoS slowloris                            → DoS
DoS Slowhttptest                         → DoS
DDoS                                     → DDoS
PortScan                                 → PortScan
FTP-Patator                              → Brute Force
SSH-Patator                              → Brute Force
Bot                                      → Bot
Web Attack � Brute Force                 → Brute Force
Web Attack � XSS                         → Web Attack
Web Attack � Sql Injection               → Web Attack
Infiltration                             → EXCLUDE
Heartbleed                               → EXCLUDE


In [ ]:
def preprocess_file(file_path):
    """
    Load and perform basic preprocessing on one CICIDS2017 CSV file.
    """

    print(f"\nProcessing: {os.path.basename(file_path)}")


    df = pd.read_csv(file_path)

    print("Original shape:", df.shape)


    df.columns = df.columns.str.strip()

   
    df = df.replace([np.inf, -np.inf], np.nan)
    before_duplicates = len(df)
    df = df.drop_duplicates()
    duplicates_removed = before_duplicates - len(df)

    df["Label"] = df["Label"].apply(map_attack_label)

    before_exclusion = len(df)
    df = df[df["Label"] != "EXCLUDE"].copy()
    excluded = before_exclusion - len(df)

    print("Duplicates removed:", duplicates_removed)
    print("Rows excluded:", excluded)
    print("Final shape:", df.shape)

    return df

In [18]:
tuesday_file = [
    f for f in files
    if "Tuesday" in os.path.basename(f)
][0]

print(tuesday_file)

../data/raw/Tuesday-WorkingHours.pcap_ISCX.csv


In [19]:
df_tuesday = preprocess_file(tuesday_file)


Processing: Tuesday-WorkingHours.pcap_ISCX.csv
Original shape: (445909, 79)
Duplicates removed: 24065
Rows excluded: 0
Final shape: (421844, 79)


In [21]:
df_tuesday["Label"].value_counts()

Label
BENIGN         412692
Brute Force      9152
Name: count, dtype: int64

In [22]:
df_tuesday["Label"].value_counts(normalize=True) * 100

Label
BENIGN         97.830478
Brute Force     2.169522
Name: proportion, dtype: float64

In [23]:
"EXCLUDE" in df_tuesday["Label"].unique()

False

In [24]:
PROCESSED_PATH = "../data/processed"

os.makedirs(PROCESSED_PATH, exist_ok=True)

In [25]:
for file in files:

    processed_df = preprocess_file(file)

    filename = os.path.basename(file)
    output_file = os.path.join(
        PROCESSED_PATH,
        filename
    )

    processed_df.to_csv(
        output_file,
        index=False
    )

    print(f"Saved → {output_file}")


Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Original shape: (288602, 79)
Duplicates removed: 35630
Rows excluded: 36
Final shape: (252936, 79)
Saved → ../data/processed/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv

Processing: Monday-WorkingHours.pcap_ISCX.csv
Original shape: (529918, 79)
Duplicates removed: 26935
Rows excluded: 0
Final shape: (502983, 79)
Saved → ../data/processed/Monday-WorkingHours.pcap_ISCX.csv

Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
Original shape: (191033, 79)
Duplicates removed: 6888
Rows excluded: 0
Final shape: (184145, 79)
Saved → ../data/processed/Friday-WorkingHours-Morning.pcap_ISCX.csv

Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Original shape: (286467, 79)
Duplicates removed: 72353
Rows excluded: 0
Final shape: (214114, 79)
Saved → ../data/processed/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv

Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Original sha